# Generalization Results Plotting

Loads results from `simu-generalization.py` (new format: `results-generalization_k{k}.pkl`).
Each file contains matrices with `L` array to filter by number of sources. Set `k_select` to plot k=3, 5, or 10.

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Results directory: MUST match simu-generalization.py (StablePCA-github/saved_results)
# Search for dir containing results-generalization_*.pkl (works regardless of cwd)
def find_results_dir():
    candidates = [
        Path.cwd() / "saved_results",
        Path.cwd().parent / "saved_results",
    ]
    for p in candidates:
        if p.exists() and (p / "results-generalization_k3_L2.pkl").exists():
            return str(p)
    return str(candidates[0])  # fallback
results_dir = find_results_dir()
print(f"Loading from: {os.path.abspath(results_dir)}")

In [ ]:
# ------------------------------------------------------------
# NeurIPS-ish global style
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "figure.dpi": 600,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "xtick.labelsize": 14,
    "ytick.labelsize": 13,
    "legend.fontsize": 15,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "legend.frameon": False,
    "lines.linewidth": 2.0,
})

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
L_list = [2, 4, 6, 8, 10]
k_list = [3, 5, 10]  # which k to load
k_select = 3  # which k to plot (change to 5 or 10 for other results)

# Methods: plot order (must match METHODS in simu-generalization.py)
# simu saves as [pooled, stable, squared, fair] -> indices 0,1,2,3
methods = ["stable", "pooled", "squared", "fair"]
method_to_col = {"pooled": 0, "stable": 1, "squared": 2, "fair": 3}
method_labels = {
    "stable":  "StablePCA",
    "pooled":  "PooledPCA",
    "squared": "SquaredPCA",
    "fair":    "FairPCA",
}
method_colors = {
    "pooled":  "#777777",
    "stable":  "#D54E00",
    "squared": "#1F78B4",
    "fair":    "#A65628",
}
method_markers = {"stable": "o", "pooled": "s", "squared": "D", "fair": "^"}

n_L = len(L_list)
n_m = len(methods)

subspace_err_mean = np.full((n_L, n_m), np.nan)
subspace_err_se   = np.zeros((n_L, n_m))
expl_src_mean = np.full((n_L, n_m), np.nan)
expl_src_se   = np.zeros((n_L, n_m))
expl_ood_mean = np.full((n_L, n_m), np.nan)
expl_ood_se   = np.zeros((n_L, n_m))

In [ ]:
# ------------------------------------------------------------
# Load results (format: one pkl per (k, L))
# ------------------------------------------------------------
n_runs_L = 0
for iL, L_val in enumerate(L_list):
    fname = os.path.join(results_dir, f"results-generalization_k{k_select}_L{L_val}.pkl")
    if not os.path.exists(fname):
        print(f"Warning: {fname} not found")
        continue
    with open(fname, "rb") as f:
        data = pickle.load(f)
    # data keys: k, L, n_runs, METHODS, seed, err, expl_var_src, expl_var_ood
    err_mat = data["err"]
    expl_src_mat = data["expl_var_src"]
    expl_ood_mat = data["expl_var_ood"]
    n_runs_L = data["n_runs"]
    for j, m in enumerate(methods):
        col = method_to_col[m]
        e = err_mat[:, col]
        s_src = expl_src_mat[:, col]
        s_ood = expl_ood_mat[:, col]
        valid_e = e[~np.isnan(e)]
        valid_src = s_src[~np.isnan(s_src)]
        valid_ood = s_ood[~np.isnan(s_ood)]
        subspace_err_mean[iL, j] = np.mean(valid_e) if len(valid_e) > 0 else np.nan
        subspace_err_se[iL, j]   = np.std(valid_e, ddof=1) / np.sqrt(len(valid_e)) if len(valid_e) > 1 else 0
        expl_src_mean[iL, j] = np.mean(valid_src) if len(valid_src) > 0 else np.nan
        expl_src_se[iL, j]   = np.std(valid_src, ddof=1) / np.sqrt(len(valid_src)) if len(valid_src) > 1 else 0
        expl_ood_mean[iL, j] = np.mean(valid_ood) if len(valid_ood) > 0 else np.nan
        expl_ood_se[iL, j]   = np.std(valid_ood, ddof=1) / np.sqrt(len(valid_ood)) if len(valid_ood) > 1 else 0

methods_with_data = [m for j, m in enumerate(methods) if np.any(~np.isnan(subspace_err_mean[:, j]))]
print(f"Loaded k={k_select}, {n_runs_L} runs per L. Methods with data: {methods_with_data}")

In [ ]:
# ------------------------------------------------------------
# Plotting
# ------------------------------------------------------------
fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(14, 4))
plt.subplots_adjust(bottom=0.23, wspace=0.25)

def plot_panel(ax, y_mean, y_se, title):
    for j, m in enumerate(methods):
        if np.all(np.isnan(y_mean[:, j])):
            continue  # skip methods with no data
        mask = ~np.isnan(y_mean[:, j])
        x_plot = [L_list[i] for i in range(len(L_list)) if mask[i]]
        y_plot = y_mean[mask, j]
        yerr_plot = y_se[mask, j]
        ax.errorbar(
            x_plot,
            y_plot,
            yerr=yerr_plot,
            label=method_labels[m],
            color=method_colors[m],
            marker=method_markers[m],
            linestyle="-",
            capsize=3,
        )
    ax.set_xlabel(r"$L$")
    ax.set_title(title, pad=10)
    ax.grid(True, linestyle=":", linewidth=0.8, alpha=0.4)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

err_title = "Shared Subspace Recovery Error" if k_select == 3 else "Subspace Capture Error"
plot_panel(ax0, subspace_err_mean, subspace_err_se, err_title)
plot_panel(ax1, expl_src_mean, expl_src_se, "Worst Exp-Var (in-dist)")
plot_panel(ax2, expl_ood_mean, expl_ood_se, "Worst Exp-Var (out-dist)")

handles, labels = ax0.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.10), ncol=4, frameon=False)
# plt.suptitle(f"$k={k_select}$", y=1.02)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()